In [5]:
import pandas as pd
from datetime import datetime

# -------------------- File paths --------------------
input_sales_path = "C:\\Users\\MOUNIKA PALLI\\Dropbox\\EBO FOLDER\\EBO SALES FOLDER\\EBO SALES DATA.xlsx"
sku_master_path = "C:\\Users\\MOUNIKA PALLI\\Dropbox\\EBO FOLDER\\MASTERS\\SKU MASTER.xlsx"
ebo_skus_path = "C:\\Users\\MOUNIKA PALLI\\Dropbox\\EBO FOLDER\\MASTERS\\EBO SKUS.xlsx"
current_stock_path = "C:\\Users\\MOUNIKA PALLI\\Dropbox\\EBO FOLDER\\STOCK\\CURRENT STOCK.xlsx"
output_excel_path = "Best_MRR_All_Stores_SKU_Level.xlsx"

# -------------------- Freebie SKUs --------------------
FREEBIE_SKUS = [
    'MABOATPB3990', 'MAAIRPODS4999', 'MAPOWBANK3999',
    'MAAIRPODS3599', 'MATROLLEY9999', 'UAGY01BLK699', 'MAKRAFTBAGBIG'
]

def exclude_freebies(df, sku_column='SKU'):
    return df[~df[sku_column].isin(FREEBIE_SKUS)].copy()

# -------------------- Calculate MRR --------------------
def calculate_sku_level_mrr(sales_df, sku_master_df):
    sales_df = sales_df.copy()
    sales_df['DATE'] = pd.to_datetime(sales_df['BILL_DATE'])
    sales_df.rename(columns={'BILL_QUANTITY': 'QTY'}, inplace=True)

    month_defs = {
        "Apr": [('W1_Apr', '2025-04-01', '2025-04-07'),
                ('W2_Apr', '2025-04-08', '2025-04-14'),
                ('W3_Apr', '2025-04-15', '2025-04-21'),
                ('W4_Apr', '2025-04-22', '2025-04-30')],
        "May": [('W1_May', '2025-05-01', '2025-05-07'),
                ('W2_May', '2025-05-08', '2025-05-14'),
                ('W3_May', '2025-05-15', '2025-05-21'),
                ('W4_May', '2025-05-22', '2025-05-28'),
                ('W5_May', '2025-05-29', '2025-05-31')],
        "June": [('W1_June', '2025-06-01', '2025-06-07'),
                 ('W2_June', '2025-06-08', '2025-06-14'),
                 ('W3_June', '2025-06-15', '2025-06-21'),
                 ('W4_June', '2025-06-22', '2025-06-30')],
        "July": [('W1_July', '2025-07-01', '2025-07-07'),
                 ('W2_July', '2025-07-08', '2025-07-14'),
                 ('W3_July', '2025-07-15', '2025-07-21'),
                 ('W4_July', '2025-07-22', '2025-07-31')]
    }

    monthly_results = {}

    for month, weeks in month_defs.items():
        month_sales = exclude_freebies(sales_df[(sales_df['DATE'] >= weeks[0][1]) & (sales_df['DATE'] <= weeks[-1][2])])
        result = pd.DataFrame()
        for label, start, end in weeks:
            week_df = month_sales[(month_sales['DATE'] >= start) & (month_sales['DATE'] <= end)]
            grouped = week_df.groupby('SKU')['QTY'].sum().reset_index()
            grouped.rename(columns={'QTY': label}, inplace=True)
            result = grouped if result.empty else pd.merge(result, grouped, on='SKU', how='outer')
        result.fillna(0, inplace=True)
        result[f'MRR_{month}'] = result[[w[0] for w in weeks]].sum(axis=1)
        monthly_results[month] = (result, weeks)

    result = monthly_results['Apr'][0]
    for month in ['May', 'June', 'July']:
        result = pd.merge(result, monthly_results[month][0], on='SKU', how='outer')
    result.fillna(0, inplace=True)

    result['MRR'] = result[['MRR_Apr', 'MRR_May', 'MRR_June', 'MRR_July']].max(axis=1)
    result['four_month_stock'] = result['MRR'] * 1.2 * 4
    result['month'] = 'August'
    result['new_mrr'] = (result['MRR'] / 12) * 2
    result['final_mrr'] = (result['MRR'] + result['new_mrr']) * 1.2

    # Ensure consistent column names from sku_master
    sku_master_df = sku_master_df.rename(columns=str.strip)
    sku_master_df = sku_master_df.rename(columns={
        'CODE': 'COLOUR',
        'size map': 'SIZE',
        'Size map': 'SIZE',
        'SIZE map': 'SIZE'
    })
    required_cols = ['SKU', 'STYLE', 'COLOUR', 'SIZE']
    for col in required_cols:
        if col not in sku_master_df.columns:
            sku_master_df[col] = ""

    result = pd.merge(result, sku_master_df[required_cols], on='SKU', how='left')
    result['STYLE*COLOUR'] = result['STYLE'].astype(str) + "*" + result['COLOUR'].astype(str)

    numeric_cols = result.select_dtypes(include='number').columns
    result[numeric_cols] = result[numeric_cols].round(0).astype('Int64')

    final_cols = ['SKU', 'STYLE', 'COLOUR', 'SIZE', 'STYLE*COLOUR']
    for month in ['Apr', 'May', 'June', 'July']:
        final_cols += [w[0] for w in month_defs[month]] + [f'MRR_{month}']
    final_cols += ['MRR', 'four_month_stock', 'month', 'new_mrr', 'final_mrr']

    return result[final_cols]

# -------------------- Load data --------------------
sales_df = pd.read_excel(input_sales_path)
sku_master_df = pd.read_excel(sku_master_path)
ebo_skus_df = pd.read_excel(ebo_skus_path)
stock_df = pd.read_excel(current_stock_path)[['SKU', 'STOCK']]

# Clean EBO SKUS column names
ebo_skus_df = ebo_skus_df.rename(columns=str.strip)
ebo_skus_df = ebo_skus_df.rename(columns={
    'Colour': 'COLOUR',
    'color': 'COLOUR',
    'COLOR': 'COLOUR',
    'SIZE map': 'SIZE',
    'Size map': 'SIZE',
    'size map': 'SIZE'
})
if 'COLOUR' not in ebo_skus_df.columns:
    ebo_skus_df['COLOUR'] = ""
if 'SIZE' not in ebo_skus_df.columns:
    ebo_skus_df['SIZE'] = ""

ebo_skus_df['STYLE*COLOUR'] = ebo_skus_df['STYLE'].astype(str) + "*" + ebo_skus_df['COLOUR'].astype(str)

# -------------------- Step 1: Calculate MRRs --------------------
mrr_df = calculate_sku_level_mrr(sales_df, sku_master_df)

# -------------------- Step 2: MRR base --------------------
mrr_base = mrr_df[['SKU', 'final_mrr', 'STYLE', 'COLOUR', 'SIZE', 'STYLE*COLOUR']].copy()
mrr_base['correct_cal'] = True
mrr_base['value'] = mrr_base['final_mrr']
mrr_base.rename(columns={'final_mrr': 'MRR'}, inplace=True)

# -------------------- Step 3: Merge with EBO SKUs --------------------
full_sku_mrr_df = pd.merge(ebo_skus_df, mrr_base[['SKU', 'MRR', 'correct_cal', 'value']], on='SKU', how='left')
full_sku_mrr_df['STYLE*COLOUR'] = full_sku_mrr_df['STYLE'].astype(str) + "*" + full_sku_mrr_df['COLOUR'].astype(str)

# -------------------- Step 4: Fill missing MRRs --------------------
def fill_missing_mrr(row, lookup_df):
    if pd.notna(row['MRR']):
        return row['MRR'], True
    match1 = lookup_df[lookup_df['STYLE*COLOUR'] == row['STYLE*COLOUR']]['MRR']
    if not match1.empty:
        return match1.iloc[0], False
    match2 = lookup_df[lookup_df['STYLE'] == row['STYLE']]['MRR']
    if not match2.empty:
        return round(match2.mean()), False
    return 0, False

full_sku_mrr_df[['MRR', 'correct_cal']] = full_sku_mrr_df.apply(
    lambda r: fill_missing_mrr(r, mrr_base), axis=1, result_type='expand'
)
full_sku_mrr_df['value'] = full_sku_mrr_df['MRR']

# -------------------- Pivot Table --------------------
pivot_df = mrr_df.pivot_table(index='STYLE*COLOUR', columns='SIZE', values='MRR', aggfunc='sum', fill_value=0).reset_index()

# -------------------- Size Mapping --------------------
size_mapping = {'8Y': 'M', 'M': 'M', '10Y': 'L', 'L': 'L', '12Y': 'XL', 'XL': 'XL', '14Y': '2XL', '2XL': '2XL'}
mapped_sizes = ['M', 'L', 'XL', '2XL']
mapped_df = pivot_df[['STYLE*COLOUR']].copy()
for target_size in mapped_sizes:
    cols_to_sum = [s for s, mapped in size_mapping.items() if mapped == target_size and s in pivot_df.columns]
    mapped_df[target_size] = pivot_df[cols_to_sum].sum(axis=1) if cols_to_sum else 0

mapped_df['COUNT'] = mapped_df[mapped_sizes].gt(0).sum(axis=1)
style_lookup = mrr_df[['STYLE*COLOUR', 'STYLE']].drop_duplicates()
mapped_df = pd.merge(mapped_df, style_lookup, on='STYLE*COLOUR', how='left')
mapped_df = mapped_df[['STYLE', 'STYLE*COLOUR'] + mapped_sizes + ['COUNT']]

# -------------------- Style Option Summary --------------------
style_option_summary = mapped_df.groupby('STYLE')['STYLE*COLOUR'].nunique().reset_index()
style_option_summary.rename(columns={'STYLE*COLOUR': 'No. of Options'}, inplace=True)

# -------------------- MOH Calculation --------------------
stock_base_df = pd.merge(
    stock_df,
    full_sku_mrr_df[['SKU', 'STYLE', 'COLOUR', 'SIZE', 'STYLE*COLOUR', 'MRR']],
    on='SKU',
    how='left'
)
stock_base_df['STOCK'] = stock_base_df['STOCK'].clip(lower=0)
stock_base_df['MOH'] = stock_base_df.apply(
    lambda row: round(row['STOCK'] / row['MRR'], 2) if row['MRR'] > 0 else None,
    axis=1
)
stock_base_df['MOH_Flag'] = stock_base_df['MOH'].apply(
    lambda x: "More than 3M" if pd.notna(x) and x > 3 else "Under 3M"
)
stock_base_df = stock_base_df[['SKU', 'STYLE', 'COLOUR', 'SIZE', 'STYLE*COLOUR', 'STOCK', 'MRR', 'MOH', 'MOH_Flag']]

# -------------------- Best MRR Summary --------------------
best_mrr_summary = mrr_df[['SKU', 'STYLE', 'COLOUR', 'SIZE', 'STYLE*COLOUR', 'MRR']].rename(columns={'MRR': 'Best_MRR'})

# -------------------- Save to Excel --------------------
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:
    mrr_df.to_excel(writer, sheet_name='SKU_MRR', index=False)
    best_mrr_summary.to_excel(writer, sheet_name='Best_MRR_Only', index=False)
    pivot_df.to_excel(writer, sheet_name='Pivot_SKU_Level_MRR', index=False)
    mapped_df.to_excel(writer, sheet_name='Mapped_Sizes_MRR', index=False)
    style_option_summary.to_excel(writer, sheet_name='Style_Options_Summary', index=False)
    full_sku_mrr_df.to_excel(writer, sheet_name='All_SKUs_With_MRR', index=False)
    stock_base_df.to_excel(writer, sheet_name='Stock_vs_MRR_MOH', index=False)

print(f"✅ Best MRR Plan across all stores saved to: {output_excel_path}")


✅ Best MRR Plan across all stores saved to: Best_MRR_All_Stores_SKU_Level.xlsx
